# gold_ancestor_coverage view

Creates (or replaces) the `genealogy.gold_ancestor_coverage` view which powers
the Ancestors tab in the research assistant app.

## What it produces
One row per direct ancestor (ancestral_proximity = 0), generations 1–6.
Columns: identity, coverage status for birth/marriage/death, boolean census
flags (1939→1841 reversed), and a DNA confirmation flag.

## Coverage status values
- `COVERED`    — original document downloaded
- `INDEX_ONLY` — index entry only, no original
- `UNCOVERED`  — no source record at all

## Census N/A logic
Done in the app frontend: grey dot shown when birth_year > census_year
or death_year < census_year − 1. The view simply returns the raw boolean flags.

## DNA
Only the `Common DNA Ancestor` tag is used (silver_person_tag + silver_tag join).
DNA Match and DNA Connection tags are for distant cousins, not direct ancestors.

## Ordering
View returns rows unordered. The app endpoint adds:
`ORDER BY generation, path_sort_key`
which groups paternal/maternal lines together within each generation.
Proper ahnentafel ordering can be added later via a Python UDF.

## To run
Run all cells once. Safe to re-run — uses CREATE OR REPLACE VIEW.
No data is written; this is a view over existing tables.

In [0]:
# ── Cell 1: Create gold_ancestor_coverage view ────────────────────────────────
# 
# Join chain:
#   gold_generation_depth          — path array, generation depth
#   gold_ancestral_proximity       — filter to direct ancestors (proximity = 0)
#   gold_person_life               — display_name, birth/death years, sex
#   gold_person_branch             — branch name
#   gold_person_fact_summary       — boolean census coverage flags
#   gold_source_coverage           — aggregated birth/marriage/death coverage status
#   silver_person_tag + silver_tag — Common DNA Ancestor tag lookup
#
# Coverage priority (best wins per event type):
#   COVERED (1) > INDEX_ONLY (2) > UNCOVERED (3)
#   MIN(status_rank) across all matching source categories gives the best status.

spark.sql("""
CREATE OR REPLACE VIEW genealogy.gold_ancestor_coverage AS

WITH

-- ── 1. Direct ancestors, generations 1-6 ─────────────────────────────────────
-- generation_depth 0 = you, 1 = parents, ..., 6 = 4x great-grandparents
-- path[0] = you, path[last] = the ancestor (confirmed from DESCRIBE output)
ancestors AS (
  SELECT
    gd.person_id                 AS person_gedcom_id,
    gd.generation_depth          AS generation,
    ARRAY_JOIN(gd.path, '|')     AS path_sort_key
  FROM genealogy.gold_generation_depth gd
  JOIN genealogy.gold_ancestral_proximity ap
    ON gd.person_id = ap.person_id
  WHERE ap.ancestral_proximity = 0
    AND gd.generation_depth BETWEEN 1 AND 6
),

-- ── 2. Coverage status ranked by quality ─────────────────────────────────────
-- Map coverage_status to a numeric rank so MIN() gives the best available status.
-- Only event-relevant source categories are considered.
coverage_ranked AS (
  SELECT
    person_gedcom_id,
    source_category,
    coverage_status,
    CASE coverage_status
      WHEN 'COVERED'    THEN 1
      WHEN 'INDEX_ONLY' THEN 2
      WHEN 'UNCOVERED'  THEN 3
      ELSE                   4
    END AS status_rank
  FROM genealogy.gold_source_coverage
  WHERE source_category IN (
    'BirthCertificate', 'BaptismRegister', 'BirthIndex',
    'MarriageCertificate', 'MarriageRegister', 'MarriageIndex',
    'DeathCertificate', 'BurialRegister', 'DeathIndex'
  )
),

-- ── 3. Aggregate to best status per event type per person ─────────────────────
coverage AS (
  SELECT
    person_gedcom_id,
    CASE MIN(CASE WHEN source_category IN ('BirthCertificate','BaptismRegister','BirthIndex')
                  THEN status_rank END)
      WHEN 1 THEN 'COVERED'
      WHEN 2 THEN 'INDEX_ONLY'
      WHEN 3 THEN 'UNCOVERED'
      ELSE        'UNCOVERED'
    END AS birth_status,
    CASE MIN(CASE WHEN source_category IN ('MarriageCertificate','MarriageRegister','MarriageIndex')
                  THEN status_rank END)
      WHEN 1 THEN 'COVERED'
      WHEN 2 THEN 'INDEX_ONLY'
      WHEN 3 THEN 'UNCOVERED'
      ELSE        'UNCOVERED'
    END AS marriage_status,
    CASE MIN(CASE WHEN source_category IN ('DeathCertificate','BurialRegister','DeathIndex')
                  THEN status_rank END)
      WHEN 1 THEN 'COVERED'
      WHEN 2 THEN 'INDEX_ONLY'
      WHEN 3 THEN 'UNCOVERED'
      ELSE        'UNCOVERED'
    END AS death_status
  FROM coverage_ranked
  GROUP BY person_gedcom_id
),

-- ── 4. DNA: Common DNA Ancestor tag only ─────────────────────────────────────
-- DNA Match (139 people) and DNA Connection (5) are for cousins/path nodes,
-- not direct ancestors. Only Common DNA Ancestor (4 people) belongs here.
dna AS (
  SELECT DISTINCT pt.person_gedcom_id, 1 AS has_dna
  FROM genealogy.silver_person_tag pt
  JOIN genealogy.silver_tag t ON pt.tag_gedcom_id = t.tag_gedcom_id
  WHERE t.tag_name = 'Common DNA Ancestor'
)

-- ── Final select ──────────────────────────────────────────────────────────────
SELECT
  a.person_gedcom_id,
  a.generation,
  a.path_sort_key,
  l.display_name,
  l.birth_year,
  l.death_year,
  l.sex,
  b.branch,

  -- Birth / Marriage / Death coverage
  COALESCE(c.birth_status,    'UNCOVERED') AS birth_status,
  COALESCE(c.marriage_status, 'UNCOVERED') AS marriage_status,
  COALESCE(c.death_status,    'UNCOVERED') AS death_status,

  -- Census boolean flags — N/A logic handled in the frontend
  -- using birth_year / death_year returned above
  COALESCE(cs.has_1841_census,   0) AS has_1841_census,
  COALESCE(cs.has_1851_census,   0) AS has_1851_census,
  COALESCE(cs.has_1861_census,   0) AS has_1861_census,
  COALESCE(cs.has_1871_census,   0) AS has_1871_census,
  COALESCE(cs.has_1881_census,   0) AS has_1881_census,
  COALESCE(cs.has_1891_census,   0) AS has_1891_census,
  COALESCE(cs.has_1901_census,   0) AS has_1901_census,
  COALESCE(cs.has_1911_census,   0) AS has_1911_census,
  COALESCE(cs.has_1921_census,   0) AS has_1921_census,
  COALESCE(cs.has_1939_register, 0) AS has_1939_register,

  -- DNA confirmation
  COALESCE(d.has_dna, 0) AS has_dna

FROM ancestors a
LEFT JOIN genealogy.gold_person_life          l  ON a.person_gedcom_id = l.person_gedcom_id
LEFT JOIN genealogy.gold_person_branch        b  ON a.person_gedcom_id = b.person_gedcom_id
LEFT JOIN genealogy.gold_person_fact_summary  cs ON a.person_gedcom_id = cs.person_gedcom_id
LEFT JOIN coverage                            c  ON a.person_gedcom_id = c.person_gedcom_id
LEFT JOIN dna                                 d  ON a.person_gedcom_id = d.person_gedcom_id
""")

print('gold_ancestor_coverage view created successfully')

In [0]:
# ── Cell 2: Smoke test — count rows by generation ─────────────────────────────
# Expected: gen 1=2, gen 2=4, gen 3=8, gen 4=16, gen 5=32, gen 6=64 (max)
# Actual counts may be lower where ancestors are unknown.

display(
  spark.sql("""
    SELECT generation, COUNT(*) AS ancestor_count,
           SUM(CASE WHEN birth_status  = 'COVERED'    THEN 1 ELSE 0 END) AS birth_covered,
           SUM(CASE WHEN marriage_status = 'COVERED'  THEN 1 ELSE 0 END) AS marr_covered,
           SUM(CASE WHEN death_status  = 'COVERED'    THEN 1 ELSE 0 END) AS death_covered,
           SUM(CASE WHEN has_dna = 1                  THEN 1 ELSE 0 END) AS dna_confirmed
    FROM genealogy.gold_ancestor_coverage
    GROUP BY generation
    ORDER BY generation
  """)
)

In [0]:
# ── Cell 3: Spot-check a generation 1 row (parents) ──────────────────────────
display(
  spark.sql("""
    SELECT *
    FROM genealogy.gold_ancestor_coverage
    WHERE generation = 1
    ORDER BY path_sort_key
  """)
)